In [ ]:
-- =============================================================================
-- SLEEPER TRADES PIPELINE
-- Dynasty-Aware Multi-Season Trade Analysis
-- 
-- Key Features:
-- - Multi-season player point tracking
-- - Draft pick career point tracking
-- - Trade completeness detection (all picks realized)
-- - Dynasty vs redraft league support
-- - Trade impact calculation across multiple time horizons
-- =============================================================================

In [ ]:
-- ---------- TRADE IMPACT AGGREGATION ----------

In [ ]:
-- NEW: Trade impact by horizon (same season, 1 year, 2 years, career)
CREATE OR REPLACE MATERIALIZED VIEW agg_trade_impact_by_horizon AS
WITH player_points AS (
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    trade_season,
    seasons_after_trade,
    SUM(points) AS player_points
  FROM fact_trade_player_points_multi_season
  GROUP BY league_id, transaction_id, side_roster_id, trade_season, seasons_after_trade
),
pick_points AS (
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    trade_season,
    seasons_after_draft AS seasons_after_trade,
    SUM(points) AS pick_points
  FROM fact_trade_pick_points_career
  GROUP BY league_id, transaction_id, side_roster_id, trade_season, seasons_after_draft
),
combined AS (
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    trade_season,
    seasons_after_trade,
    COALESCE(player_points, 0) AS player_points,
    0 AS pick_points
  FROM player_points
  
  UNION ALL
  
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    trade_season,
    seasons_after_trade,
    0 AS player_points,
    COALESCE(pick_points, 0) AS pick_points
  FROM pick_points
),
aggregated AS (
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    trade_season,
    seasons_after_trade,
    SUM(player_points) AS player_points,
    SUM(pick_points) AS pick_points,
    SUM(player_points) + SUM(pick_points) AS total_points
  FROM combined
  GROUP BY league_id, transaction_id, side_roster_id, trade_season, seasons_after_trade
)
SELECT
  a.*,
  -- Add metadata
  dm.league_type,
  dm.cluster_key,
  dm.cluster_name
FROM aggregated a
LEFT JOIN dim_trade_metadata dm
  ON a.league_id = dm.league_id
  AND a.transaction_id = dm.transaction_id;

In [ ]:
-- NEW: Trade impact summary with multiple time horizons
CREATE OR REPLACE MATERIALIZED VIEW agg_trade_impact_summary AS
WITH horizons AS (
  SELECT
    league_id,
    transaction_id,
    side_roster_id,
    trade_season,
    league_type,
    cluster_key,
    cluster_name,
    -- Same season impact (seasons_after_trade = 0)
    SUM(CASE WHEN seasons_after_trade = 0 THEN total_points ELSE 0 END) AS same_season_points,
    -- Year 1 after trade
    SUM(CASE WHEN seasons_after_trade = 1 THEN total_points ELSE 0 END) AS year1_points,
    -- Year 2 after trade
    SUM(CASE WHEN seasons_after_trade = 2 THEN total_points ELSE 0 END) AS year2_points,
    -- Year 3+ after trade
    SUM(CASE WHEN seasons_after_trade >= 3 THEN total_points ELSE 0 END) AS year3plus_points,
    -- Total career impact
    SUM(total_points) AS career_points
  FROM agg_trade_impact_by_horizon
  GROUP BY league_id, transaction_id, side_roster_id, trade_season, league_type, cluster_key, cluster_name
)
SELECT
  h.*,
  tc.is_complete AS trade_is_complete,
  tc.total_picks,
  tc.realized_picks
FROM horizons h
LEFT JOIN dim_trade_completeness tc
  ON h.league_id = tc.league_id
  AND h.transaction_id = tc.transaction_id;

In [ ]:
-- NEW: Head-to-head trade comparison (who won the trade?)
CREATE OR REPLACE MATERIALIZED VIEW agg_trade_winners AS
WITH roster_pairs AS (
  SELECT DISTINCT
    league_id,
    transaction_id,
    side_roster_id
  FROM agg_trade_impact_summary
),
paired AS (
  SELECT
    a.league_id,
    a.transaction_id,
    a.side_roster_id AS roster_a,
    b.side_roster_id AS roster_b
  FROM roster_pairs a
  JOIN roster_pairs b
    ON a.league_id = b.league_id
    AND a.transaction_id = b.transaction_id
    AND a.side_roster_id < b.side_roster_id  -- Avoid duplicate pairs
),
with_scores AS (
  SELECT
    p.*,
    sa.career_points AS roster_a_points,
    sb.career_points AS roster_b_points,
    sa.trade_is_complete,
    sa.trade_season,
    sa.league_type,
    sa.cluster_key,
    sa.cluster_name
  FROM paired p
  JOIN agg_trade_impact_summary sa
    ON p.league_id = sa.league_id
    AND p.transaction_id = sa.transaction_id
    AND p.roster_a = sa.side_roster_id
  JOIN agg_trade_impact_summary sb
    ON p.league_id = sb.league_id
    AND p.transaction_id = sb.transaction_id
    AND p.roster_b = sb.side_roster_id
)
SELECT
  league_id,
  transaction_id,
  trade_season,
  league_type,
  cluster_key,
  cluster_name,
  roster_a,
  roster_b,
  roster_a_points,
  roster_b_points,
  roster_a_points - roster_b_points AS point_differential,
  CASE
    WHEN roster_a_points > roster_b_points THEN roster_a
    WHEN roster_b_points > roster_a_points THEN roster_b
    ELSE NULL  -- Tie
  END AS winner_roster_id,
  CASE
    WHEN roster_a_points < roster_b_points THEN roster_a
    WHEN roster_b_points < roster_a_points THEN roster_b
    ELSE NULL  -- Tie
  END AS loser_roster_id,
  ABS(roster_a_points - roster_b_points) AS trade_impact_magnitude,
  trade_is_complete
FROM with_scores;

In [ ]:
-- ---------- ENRICHED VIEWS WITH NAMES ----------

In [ ]:
-- Trade winners with manager names
CREATE OR REPLACE MATERIALIZED VIEW agg_trade_winners_enriched AS
SELECT
  tw.*,
  ma.manager_display_name AS roster_a_manager,
  mb.manager_display_name AS roster_b_manager,
  mw.manager_display_name AS winner_manager,
  ml.manager_display_name AS loser_manager
FROM agg_trade_winners tw
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map ma
  ON tw.league_id = ma.league_id
  AND tw.roster_a = ma.roster_id
  AND tw.trade_season = ma.season
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mb
  ON tw.league_id = mb.league_id
  AND tw.roster_b = mb.roster_id
  AND tw.trade_season = mb.season
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map mw
  ON tw.league_id = mw.league_id
  AND tw.winner_roster_id = mw.roster_id
  AND tw.trade_season = mw.season
LEFT JOIN workspace.sleeper_core.dim_manager_roster_map ml
  ON tw.league_id = ml.league_id
  AND tw.loser_roster_id = ml.roster_id
  AND tw.trade_season = ml.season;